In [23]:
!pip install catboost
!pip install lightgbm
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 8.8 MB/s eta 0:00:00


In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
from tqdm import tqdm
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [13]:
df = pd.read_csv("/content/train.csv",index_col=0)
df

,Negara/Tahun,Emisi Savanna Api,Emisi Kebakaran Hutan,Emisi Residu Tanaman,Emisi Budidaya Padi,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,...,Emisi Manajemen Pupuk,Emisi Kebakaran Di Tanah Organik,Emisi Kebakaran Di Hutan Tropis Yang Lembab,Penggunaan Energi Di Pertanian,Populasi Pedesaan,Populasi Perkotaan,Total Populasi - Pria,Total Populasi - Wanita,Emisi Total,Peningkatan Suhu Rata - Rata ° C
0,Afghanistan/1990,14.7237,0.0557,205.6077,686.0000,0.0,11.807483,63.1152,-2388.8030,0.0000,...,319.1763,0.0,0.0,NaN,9655167.0,2593947.0,5348387.0,5346409.0,2198.963539,0.536167
1,Afghanistan/1991,14.7237,0.0557,209.4971,678.1600,0.0,11.712073,61.2125,-2388.8030,0.0000,...,342.3079,0.0,0.0,NaN,10230490.0,2763167.0,5372959.0,5372208.0,2323.876629,0.020667
2,Afghanistan/1992,14.7237,0.0557,196.5341,686.0000,0.0,11.712073,53.3170,-2388.8030,0.0000,...,349.1224,0.0,0.0,NaN,10995568.0,2985663.0,6028494.0,6028939.0,2356.304229,-0.259583
3,Afghanistan/1993,14.7237,0.0557,230.8175,686.0000,0.0,11.712073,54.3617,-2388.8030,0.0000,...,352.2947,0.0,0.0,NaN,11858090.0,3237009.0,7003641.0,7000119.0,2368.470529,0.101917
4,Afghanistan/1994,14.7237,0.0557,242.0494,705.6000,0.0,11.712073,53.9874,-2388.8030,0.0000,...,367.6784,0.0,0.0,NaN,12690115.0,3482604.0,7733458.0,7722096.0,2500.768729,0.372250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5598,Zimbabwe/2010,2795.6192,283.6316,109.3055,3.0243,0.0,87.000000,157.6366,262.6108,10677.6439,...,302.3508,0.0,0.0,2074.1869,9410211.0,4676106.0,6034165.0,6805605.0,23601.395453,0.911917
5599,Zimbabwe/2011,2918.2098,168.1659,100.7798,3.7456,0.0,94.000000,268.6740,648.0808,10670.8870,...,312.2478,0.0,0.0,966.3719,9636932.0,4749717.0,6114111.0,6911674.0,23470.631605,0.191167
5600,Zimbabwe/2012,2164.8953,259.4249,103.8422,5.5527,0.0,91.000000,304.6578,648.0808,10670.8870,...,314.3433,0.0,0.0,908.2629,9880721.0,4830105.0,6223803.0,7041528.0,22903.232305,0.337000
5601,Zimbabwe/2013,1544.9329,238.1898,96.8978,5.8016,0.0,73.000000,338.5506,648.0808,10670.8870,...,317.9359,0.0,0.0,546.0138,10138667.0,4915839.0,6363142.0,7192279.0,22245.497305,0.089667


In [14]:
# Memisahkan kolom "Negara/Tahun" menjadi "Negara" dan "Tahun"
df[['Negara', 'Tahun']] = df['Negara/Tahun'].str.split('/', expand=True)
df['Tahun'] = df['Tahun'].astype(int)
df = df.drop(['Negara/Tahun'], axis=1)

In [15]:
print("Kolom 'Tahun' pada df:\n", df[['Tahun']].head())

Kolom 'Tahun' pada df:
    Tahun
0   1990
1   1991
2   1992
3   1993
4   1994


In [19]:
def regressor_imputer(df, feature, max_depth=6):
    """Fungsi untuk mengimputasi nilai yang hilang pada 'feature'
       menggunakan prediksi RandomForestRegressor."""
    df_filled = df.copy()
    if df_filled[feature].isna().any():
        missing_data = df_filled[df_filled[feature].isna()]
        non_missing_data = df_filled.dropna(subset=[feature])

        X_train = non_missing_data.drop(columns=[feature])
        y_train = non_missing_data[feature]

        imputer = SimpleImputer()
        X_train_imputed = imputer.fit_transform(X_train)
        X_missing = missing_data.drop(columns=[feature])
        X_missing_imputed = imputer.transform(X_missing)

        rf = RandomForestRegressor(max_depth=max_depth)
        rf.fit(X_train_imputed, y_train)

        y_missing_pred = rf.predict(X_missing_imputed)
        df_filled.loc[df_filled[feature].isna(), feature] = y_missing_pred
    return df_filled

In [20]:
# Gunakan df (yang sudah dimodifikasi) untuk menentukan fitur numerik dan kategorikal
var_numerik = [col for col in df.columns if df[col].dtype in ["int64", "float64"]]
var_categorik = [col for col in df.columns if df[col].dtype == "object"]

# Buat daftar fitur numerik yang memiliki missing value
missing_values = df[var_numerik].isna().sum()
missing_list = missing_values[missing_values > 0].index.tolist()

# Ambil subset DataFrame hanya dengan fitur numerik
numerik_aja = df[var_numerik]

def replace_missing(df, missing_list):
    numeric_df = df.copy()
    for feature in tqdm(missing_list, desc="Mengimputasi nilai hilang"):
        numeric_df = regressor_imputer(numeric_df, feature)
    return numeric_df

num_df = replace_missing(numerik_aja, missing_list)

# Proses fitur kategorikal: lakukan label encoding
cat_df = df[var_categorik].copy()
label_encoder = LabelEncoder()
for column in cat_df.columns:
    cat_df[column] = label_encoder.fit_transform(cat_df[column].astype(str))
    # Pastikan hasilnya berupa numeric
    cat_df[column] = pd.to_numeric(cat_df[column], errors='coerce')

Mengimputasi nilai hilang: 100%|██████████| 11/11 [01:05<00:00,  6.00s/it]


In [32]:
# Gabungkan DataFrame numerik dan kategorikal menjadi final_df
final_df = pd.concat([cat_df, num_df], axis=1)

# Pastikan final_df memiliki kolom "Tahun"
if "Tahun" not in final_df.columns:
    raise ValueError("Kolom 'Tahun' tidak ditemukan di final_df!")
else:
    print("final_df sudah memiliki kolom 'Tahun'.")

final_df sudah memiliki kolom 'Tahun'.


In [33]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor, BaggingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, Ridge
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error

In [34]:
final_df = pd.concat([cat_df,num_df], axis =1)

def df_split(final_df, tahun_cutoff=2013):
    df_copy = final_df.copy()
    x_train = df_copy.loc[df_copy['Tahun'] < tahun_cutoff]
    y_train = x_train["Peningkatan Suhu Rata - Rata ° C"]
    x_train = x_train.drop(columns=["Peningkatan Suhu Rata - Rata ° C"])

    x_test = df_copy.loc[df_copy['Tahun'] >= tahun_cutoff]
    y_test = x_test["Peningkatan Suhu Rata - Rata ° C"]
    x_test = x_test.drop(columns=["Peningkatan Suhu Rata - Rata ° C"])
    return x_train, y_train, x_test, y_test

In [35]:
x_train,y_train,x_test,y_test = df_split(final_df)

In [36]:
from catboost import CatBoostRegressor

In [37]:


def model_evaluation(x_train,x_test,y_train,y_test):
    models = {

        "LGBMRegressor":LGBMRegressor(),
        "KNeighborsRegressor" : KNeighborsRegressor(),
        "RandomForestRegressor": RandomForestRegressor(),
        "GradientBoostingRegressor": GradientBoostingRegressor(),
        "BaggingRegressor":BaggingRegressor(),
        "XGBRegressor": XGBRegressor(),
        "XGBRegressor": CatBoostRegressor(verbose=0)
    }

    for model_name, model in models.items():
        model.fit(x_train, y_train)
        test_pred = model.predict(x_test)
        folds = KFold(n_splits=5)

        mae = mean_absolute_error(y_test, test_pred)
        mse = mean_squared_error(y_test, test_pred)
        mape = mean_absolute_percentage_error(y_test, test_pred)
        cv_score=cross_val_score(model,x_train,y_train,cv=folds, scoring = "neg_mean_absolute_error")
        cv_score = np.mean(cv_score)

        print(model_name)
        print("MAPE:", round(mape, 4))
        print("MAE:", round(mae, 4))
        print("MSE:", round(mse, 4))
        # print("Cross_val_score", cv_score)
        print("------------"*3)

model_evaluation(x_train,x_test,y_train,y_test)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000964 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7146
[LightGBM] [Info] Number of data points in the train set: 5149, number of used features: 30
[LightGBM] [Info] Start training from score 0.724201
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000725 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7108
[LightGBM] [Info] Number of data points in the train set: 4119, number of used features: 30
[LightGBM] [Info] Start training from score 0.725370
[LightGBM] [Warning] Found whi